In [8]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from youtube_search import YoutubeSearch
import json

@tool
def search_youtube_detailed(query: str, max_results: int = 15) -> str:
    """유튜브에서 검색어로 영상을 찾아 제목·채널·조회수·업로드일·설명을 조회수 순으로 반환한다."""
    results = YoutubeSearch(query, max_results=max_results).to_dict()
    videos = [
        {
            "title": r["title"],
            "channel": r["channel"],
            "views": r["views"],
            "publish_time": r["publish_time"],
            "duration": r["duration"],
            "url": f"https://youtube.com{r['url_suffix']}",
        }
        for r in results
    ]
    # 조회수 문자열("1.2만회" 등)에서 숫자만 뽑아 정렬 — 아래 정렬 셀 참고
    return json.dumps(videos, ensure_ascii=False)

In [10]:
import re

def parse_views(view_str: str) -> int:
    """'1.2만회', '3,456회' 같은 문자열을 정수로 변환."""
    num = re.search(r"[\d,\.]+", view_str)
    if not num:
        return 0
    value = float(num.group().replace(",", ""))
    if "만" in view_str:
        value *= 10000
    elif "억" in view_str:
        value *= 100000000
    return int(value)

result = search_youtube_detailed.invoke({"query": "옥씨부인전 몰아보기", "max_results": 30})
videos = json.loads(result)
videos_sorted = sorted(videos, key=lambda v: parse_views(v["views"]), reverse=True)

for v in videos_sorted:
    print(f"{v['views']:>10} | {v['channel']:<15} | {v['title']}")

조회수 1,480,116회 | 고몽              | 임지연진이는 잊어라..! 이젠 '노비 연진이'다! 초고지능IQ를 타고난 천한 신분이 말빨과 논리로 배우신 양반들 깨부수며 신분급상승하는 쾌감 개쩌는 드라마 [옥씨부인전] JTBC
조회수 868,381회 | DRAMA Voyage    | (2시간) IQ 200 천재 노비가 '최초 여성 변호사'로 🔥초고속 신분 상승🔥하면 생기는 일...ㅎ｜옥씨부인전 | JTBC 241229 방송 외
조회수 461,217회 | JTBC Drama      | [13회 하이라이트] 서로의 비밀을 손에 쥔 도망 노비 임지연-사기꾼 호판대감 최정우💥 두 사람의 팽팽한 대면‍🔥 | 옥씨부인전 | JTBC 250118 방송
조회수 426,016회 | DRAMA Voyage    | 🔥통쾌🔥 ＂첩이 본부인 행세라뇨＂ 임지연 정체 밝히려 온 소혜 아씨 말로 패는 자모회장 윤지혜ㅋ｜옥씨부인전 | JTBC 250118 방송
조회수 404,233회 | JTBC Drama      | [5-6회 하이라이트] 시동생 장원급제까지 시키며 집안을 일으킨 부인을 7년 만에 만났는데.. 외면하는 추영우 모습에 임지연 오열 | 옥씨부인전 | JTBC 241222 방송
조회수 291,820회 | DRAMA Voyage    | 【7~최종화 몰아보기🎬】 💥사이다💥 10년 만에 잡은 도망노비가 하필 최초 여성 변호사💦 천재 노비가 학대아씨 응징하는 법? | 옥씨부인전 | JTBC 250126 방송 외
조회수 285,816회 | DRAMA Voyage    | (2시간) IQ 200 법잘알 도망 노비가 학대 아씨에게 복수하고 벗어나는 법♨｜옥씨부인전 | JTBC 250126 방송 외
조회수 263,665회 | DRAMA Voyage    | 임지연X추영우(천승휘) 통합본📂 독박육아에 처가살이까지..💦 바깥일 뒷바라지 몰라주는 부인 잔소리에 삐친 서방님ㅋㅋ｜옥씨부인전｜JTBC 250126 방송 외
조회수 258,397회 | DRAMA Voyage    | 

In [11]:
from IPython.display import display, HTML

html = "<table>"
for v in videos_sorted:
    html += f"<tr><td>{v['views']}</td><td>{v['channel']}</td><td><a href='{v['url']}' target='_blank'>{v['title']}</a></td></tr>"
html += "</table>"

display(HTML(html))

"조회수 1,480,116회",고몽,임지연진이는 잊어라..! 이젠 '노비 연진이'다! 초고지능IQ를 타고난 천한 신분이 말빨과 논리로 배우신 양반들 깨부수며 신분급상승하는 쾌감 개쩌는 드라마 [옥씨부인전] JTBC
"조회수 868,381회",DRAMA Voyage,(2시간) IQ 200 천재 노비가 '최초 여성 변호사'로 🔥초고속 신분 상승🔥하면 생기는 일...ㅎ｜옥씨부인전 | JTBC 241229 방송 외
"조회수 461,217회",JTBC Drama,[13회 하이라이트] 서로의 비밀을 손에 쥔 도망 노비 임지연-사기꾼 호판대감 최정우💥 두 사람의 팽팽한 대면‍🔥 | 옥씨부인전 | JTBC 250118 방송
"조회수 426,016회",DRAMA Voyage,🔥통쾌🔥 ＂첩이 본부인 행세라뇨＂ 임지연 정체 밝히려 온 소혜 아씨 말로 패는 자모회장 윤지혜ㅋ｜옥씨부인전 | JTBC 250118 방송
"조회수 404,233회",JTBC Drama,[5-6회 하이라이트] 시동생 장원급제까지 시키며 집안을 일으킨 부인을 7년 만에 만났는데.. 외면하는 추영우 모습에 임지연 오열 | 옥씨부인전 | JTBC 241222 방송
"조회수 291,820회",DRAMA Voyage,【7~최종화 몰아보기🎬】 💥사이다💥 10년 만에 잡은 도망노비가 하필 최초 여성 변호사💦 천재 노비가 학대아씨 응징하는 법? | 옥씨부인전 | JTBC 250126 방송 외
"조회수 285,816회",DRAMA Voyage,(2시간) IQ 200 법잘알 도망 노비가 학대 아씨에게 복수하고 벗어나는 법♨｜옥씨부인전 | JTBC 250126 방송 외
"조회수 263,665회",DRAMA Voyage,임지연X추영우(천승휘) 통합본📂 독박육아에 처가살이까지..💦 바깥일 뒷바라지 몰라주는 부인 잔소리에 삐친 서방님ㅋㅋ｜옥씨부인전｜JTBC 250126 방송 외
"조회수 258,397회",DRAMA Voyage,임지연X윤지혜 통합본📂 ＂괴질 증상 아십니까?＂ 소혜 아씨 당황ㅋㅋ;; 무기력한 노비 🔥각성 max🔥 시키는 마님의 방문｜옥씨부인전｜JTBC 250126 방송 외
"조회수 187,402회",JTBC Drama,[스페셜] 이번 생엔 외과 의사로 태어나 행복하길 바라..🥺 가족 외면했던 대가를 목숨 바쳐 갚은 '진짜 서방님' 추영우 서사 | 옥씨부인전 | JTBC 250126 방송
"조회수 167,744회",DRAMA Voyage,이런 순애보는 첨이야...💗 7년 전 떠난 남편인 척 살겠다는 짝남의 고백｜추영우 명대사｜옥씨부인전｜JTBC 250112 방송 외


In [12]:
from langchain_community.document_loaders import YoutubeLoader

@tool
def get_youtube_transcript(video_url: str) -> str:
    """유튜브 영상 URL에서 자막(스크립트)을 텍스트로 가져온다. 자막이 없는 영상은 오류를 반환한다."""
    try:
        loader = YoutubeLoader.from_youtube_url(video_url, language=["ko", "en"])
        docs = loader.load()
        if not docs:
            return "자막을 찾을 수 없습니다."
        text = docs[0].page_content
        return text[:2000]  # 절단 — 비용 관리 목적
    except Exception as e:
        return f"자막 가져오기 오류: {e}"

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_10896\3423232476.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import YoutubeLoader


In [13]:
first_video_url = videos_sorted[0]["url"]
transcript = get_youtube_transcript.invoke({"video_url": first_video_url})
print(transcript[:500])

자막 가져오기 오류: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=PRtEPcpgW1c! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

Ways to wo


In [14]:
second_video_url = videos_sorted[1]["url"]
print("시도할 영상:", videos_sorted[1]["title"])

transcript = get_youtube_transcript.invoke({"video_url": second_video_url})
print(transcript[:500])

시도할 영상: (2시간) IQ 200 천재 노비가 '최초 여성 변호사'로 🔥초고속 신분 상승🔥하면 생기는 일...ㅎ｜옥씨부인전 | JTBC 241229 방송 외
자막 가져오기 오류: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=EiQmvQRDZ9g! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

Ways to wo


In [15]:
video_summary_text = "\n".join(
    f"{v['views']} | {v['channel']} | {v['title']}" for v in videos_sorted
)
print(video_summary_text[:500])

조회수 1,480,116회 | 고몽 | 임지연진이는 잊어라..! 이젠 '노비 연진이'다! 초고지능IQ를 타고난 천한 신분이 말빨과 논리로 배우신 양반들 깨부수며 신분급상승하는 쾌감 개쩌는 드라마 [옥씨부인전] JTBC
조회수 868,381회 | DRAMA Voyage | (2시간) IQ 200 천재 노비가 '최초 여성 변호사'로 🔥초고속 신분 상승🔥하면 생기는 일...ㅎ｜옥씨부인전 | JTBC 241229 방송 외
조회수 461,217회 | JTBC Drama | [13회 하이라이트] 서로의 비밀을 손에 쥔 도망 노비 임지연-사기꾼 호판대감 최정우💥 두 사람의 팽팽한 대면‍🔥 | 옥씨부인전 | JTBC 250118 방송
조회수 426,016회 | DRAMA Voyage | 🔥통쾌🔥 ＂첩이 본부인 행세라뇨＂ 임지연 정체 밝히려 온 소혜 아씨 말로 패는 자모회장 윤지혜ㅋ｜옥씨부인전 | JTBC 250118 방송
조회수 404,233회 | JTBC Drama | [5-6회 하이라이트] 시


In [16]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 콘텐츠 재생산 양상을 분석하는 연구 보조원입니다. 주어진 유튜브 검색 결과 목록을 보고, 채널 유형별로 어떤 재해석/재편집 경향이 있는지 간단히 요약하세요."),
    ("human", "{video_list}"),
])

summarize_chain = summarize_prompt | model | StrOutputParser()

summary = summarize_chain.invoke({"video_list": video_summary_text})
print(summary)

제시된 유튜브 검색 결과를 바탕으로 드라마 《옥씨부인전》 관련 콘텐츠의 **채널 유형별 재해석 및 재편집 경향**을 분석한 요약 보고입니다.

---

### 1. 영화/드라마 전문 크리에이터 채널 (예: 고몽, 리뷰 봐야지)
* **주요 경향:** 배우의 전작 이미지를 활용한 강렬한 어그로(Hook)와 스피디한 압축 요약
* **재해석 방식:** 
  * 배우 임지연의 대표작(더 글로리) 속 캐릭터를 언급(**"연진이는 잊어라! 이젠 노비 연진이다"**)하여 직관적인 재미와 친숙함을 유발합니다.
  * 초고지능(IQ 200), 법률 지식, 신분 상승, 통쾌한 말빨 등 **웹소설적 쾌감 요소(사이다·능력자물)**를 극대화하여 서사를 재구성합니다.
* **특징:** 검색 결과 중 최고 조회수(148만 회)를 기록하며, 드라마의 초반 유입을 이끄는 가장 강력한 확장력을 보입니다.

---

### 2. 방송사 큐레이션/아카이브 채널 (예: DRAMA Voyage, 또 Voyage)
* **주요 경향:** 장시간(2~5시간) 롱폼 몰아보기 및 테마/관계성 중심의 세분화된 편집
* **재해석 방식:**
  * **테마형 편집:** ‘복수극’, ‘신분 상승’, ‘통쾌한 사이다’ 등 시청자가 선호하는 쾌감 중심의 키워드로 장면들을 모아 재편집합니다.
  * **관계성 및 캐릭터 중심:** 특정 인물 조합(임지연X추영우, 임지연X윤지혜)이나 특정 서사(짝남의 순애보, 남편의 등장 씬 등)만 추출하여 팬덤 맞춤형 콘텐츠로 재가공합니다.
* **특징:** 회차별 몰아보기(1~6화, 7~최종화 등)와 2시간 이상의 장시간 영상 비중이 높아, 시청자의 체류 시간을 극대화하는 밥친구/정주행용 콘텐츠로 활용됩니다.

---

### 3. 방송사 공식 메인 채널 (예: JTBC Drama)
* **주요 경향:** 최신 회차의 핵심 클라이맥스 하이라이트 및 특정 인물 스페셜
* **재해석 방식:**
  * 방송 직후 주요 갈등 장면(비밀을 쥔 대면 씬, 감정적 오열 씬 등)을 짧게 편집

In [17]:
test_video_url = "https://www.youtube.com/watch?v=ADqTvjZUlqg"  # TED-Ed 예시

transcript = get_youtube_transcript.invoke({"video_url": test_video_url})
print(transcript[:500])

자막 가져오기 오류: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=ADqTvjZUlqg! This is most likely caused by:

The video is no longer available

If you are sure that the described cause is not responsible for this error and that a transcript should be retrievable, please create an issue at https://github.com/jdepoix/youtube-transcript-api/issues. Please add which version of youtube_transcript_api you are using and provide the information needed to replicate the error. Al


In [18]:
edu_search = search_youtube.invoke({"query": "TED-Ed", "max_results": 3})
print(edu_search)

[{"title": "Why is being scared so fun? - Margee Kerr", "url": "https://youtube.com/watch?v=oetVvR5RQUs&pp=ygUGVEVELUVk"}, {"title": "What staying up all night does to your brain - Anna Rothschild", "url": "https://youtube.com/watch?v=idrbwnWLJ7w&pp=ygUGVEVELUVk"}, {"title": "과자 속의 화학 - 스테파니 워렌 (Stephanie Warren)", "url": "https://youtube.com/watch?v=n6wpNhyreDE&pp=ygUGVEVELUVk"}]


In [19]:
import json
edu_videos = json.loads(edu_search)
test_video_url = edu_videos[0]["url"]
print("시도할 영상:", edu_videos[0]["title"])

transcript = get_youtube_transcript.invoke({"video_url": test_video_url})
print(transcript[:500])

시도할 영상: Why is being scared so fun? - Margee Kerr
자막 가져오기 오류: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=oetVvR5RQUs! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

Ways to wo


In [20]:
# multiturn_chatbot.py
import os
from dotenv import load_dotenv
from datetime import datetime
from zoneinfo import ZoneInfo

from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
from langchain_community.document_loaders import YoutubeLoader
from youtube_search import YoutubeSearch

load_dotenv()


@tool
def get_word_count(text: str) -> str:
    """텍스트의 단어 수와 글자 수를 센다."""
    return f"단어 수: {len(text.split())}, 글자 수: {len(text)}"


@tool
def get_current_time_with_timezone(timezone: str) -> str:
    """IANA 시간대 이름(예: 'Asia/Seoul', 'America/New_York')을 받아
    해당 지역의 현재 시간을 알려준다."""
    try:
        now = datetime.now(ZoneInfo(timezone))
        return now.strftime("%Y-%m-%d %H:%M:%S %Z")
    except Exception as e:
        return f"시간대 오류: {e} (예: 'Asia/Seoul', 'America/New_York' 형식으로 입력)"


@tool
def get_web_search(query: str, search_period: str = "m") -> str:
    """웹에서 최신 정보를 검색한다.

    Args:
        query: 검색어
        search_period: 검색 기간 ('d': 하루, 'w': 일주일, 'm': 한 달, 'y': 일 년)
    """
    wrapper = DuckDuckGoSearchAPIWrapper(region="kr-kr", time=search_period)
    search = DuckDuckGoSearchResults(api_wrapper=wrapper, results_separator=";\n")
    return search.invoke(query)


@tool
def get_youtube_search(query: str) -> str:
    """유튜브에서 영상을 검색하고, 각 영상의 자막 내용까지 가져온다.
    영상 추천·비교·내용 요약 요청에 사용한다.

    Args:
        query: 검색어
    """
    videos = YoutubeSearch(query, max_results=3).to_dict()
    videos = [v for v in videos if len(v["duration"].split(":")) < 3]  # 1시간 이상 제외

    results = []
    for v in videos:
        video_url = "https://youtube.com" + v["url_suffix"].split("&")[0]
        try:
            docs = YoutubeLoader.from_youtube_url(video_url, language=["ko", "en"]).load()
            content = " ".join(d.page_content for d in docs)[:2000]
        except Exception as error:
            content = f"(자막을 가져오지 못했습니다: {error})"
        results.append(
            f"제목: {v['title']}\n채널: {v['channel']} | 길이: {v['duration']}\n"
            f"URL: {video_url}\n자막(일부): {content}"
        )
    return "\n\n---\n\n".join(results) if results else "검색 결과가 없습니다."


tools = [
    get_word_count,
    get_current_time_with_timezone,
    get_web_search,
    get_youtube_search,
]
tool_map = {t.name: t for t in tools}

MODEL = "google_genai:gemini-flash-latest"   # 오늘 실습에서 확인된 모델
model = init_chat_model(MODEL, temperature=0)
model_with_tools = model.bind_tools(tools)


def get_ai_response(messages: list) -> list:
    """모델을 호출하고, 도구 호출이 있으면 실행까지 마친 뒤
    최종 답변까지 포함된 전체 messages를 반환한다."""
    ai_message = model_with_tools.invoke(messages)
    messages.append(ai_message)

    max_rounds = 5  # 도전 과제 1: 반복 횟수 상한
    rounds = 0
    while ai_message.tool_calls and rounds < max_rounds:
        for call in ai_message.tool_calls:
            tool_fn = tool_map[call["name"]]
            result = tool_fn.invoke(call["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
        ai_message = model_with_tools.invoke(messages)
        messages.append(ai_message)
        rounds += 1

    return messages


def main():
    print("=== 도구 4개를 가진 멀티턴 챗봇 (종료하려면 'exit' 입력) ===")
    messages = [
        SystemMessage(content=(
            "너는 사용자의 질문에 가장 적합한 도구를 골라 호출해 답하는 AI 조교다. "
            "도구 결과에 없는 내용은 추측하지 마라."
        ))
    ]

    while True:
        user_input = input("\n나: ").strip()
        if user_input.lower() in ("exit", "quit", "종료"):
            print("챗봇을 종료합니다.")
            break
        if not user_input:
            continue

        messages.append(HumanMessage(content=user_input))
        messages = get_ai_response(messages)

        print("에이전트:", messages[-1].content)


if __name__ == "__main__":
    main()


=== 도구 4개를 가진 멀티턴 챗봇 (종료하려면 'exit' 입력) ===



나:  지금 서울이랑 뉴욕 몇시야?


에이전트: [{'type': 'text', 'text': '현재 서울과 뉴욕의 시각은 다음과 같습니다.\n\n* **서울**: 2026년 8월 10일 오후 4시 10분 (KST)\n* **뉴욕**: 2026년 8월 10일 오전 3시 10분 (EDT)', 'extras': {'signature': 'EvwBCvkBARFNMg+sEnjBKb2Sn+ETdGKNpBkx69FE5LN0zDSnLFM3sAgFhxDfKQhn5V77O2R1dM0zLJUzBm5ASMMrhvckILwcIEtC3Ty8XvJFGxsySAkjQnRd3F+UK4KupeGglDk9F+/JlTC3RZafFCFAEmqJ/chgQrIai9KM+8S5ZPnPgjcC2Q9rlfN9zswps20Cfmy5eD+KniAXRcOSr5au8nYBi2F82zmfW+MKF2fiP7cCIfwPlORX4+s1qOyzbedfovma3WQYVH7D5hqBVBteFo6LXwKPHc7my+r4xQ9ZQUi5ywCxuam7Lq4NbJ8jEuMTyV2V/paG3yobEkug'}}]



나:  옥씨부인전 유튜브 중에 가장 영향력이 있는 영상은 뭐야?


에이전트: [{'type': 'text', 'text': "현재 유튜브 검색 결과에서 '옥씨부인전' 관련 영상 데이터를 확인할 수 없어 가장 영향력이 있는 영상을 안내해 드리기 어렵습니다.", 'extras': {'signature': 'EtgCCtUCARFNMg+RPa4KI5vk/RCVTozcLYkNmvpzt9KemBe8MEiQtLQ3ebLLJEHveG1KoZGJeFXLv/cKgSb5S4TvUdDhSAjxuOBfe2CyCcX/MVxNm864Mwy6PKRXwCoiUm8oTye5KwCLtqD2ydjiq0tZXfN/Um1AK4w010jipAa1nQLb4AC80cKM4K+xsyq8hR6eIjoQir86inwxYerWI0kp8QkVdrnHRoJlABoPUMdh9/O8SvdY37WAKTPNw2zffxAjyb6APbFPX4i2jJMejuKpkUgMqapwaU2ggGsz+JF3qbG6T5Jey6F3xgsTh+FuCj3ebrI/hTxAN+Mtt9/rmc8Y19ZAuVo2A0+4+8P2Fn2Fbm8HDThBDOY8XY0KSkdebDNR9RtgsYvPYjPUO4TvMEJUpLIeE1NCNWrTHSas11JXqO2n7uN3Mi7+stQRSq8M7oYuPVu20pbGoMY='}}]



나:  옥씨부인전 리뷰 영상 찾아서 내용 요약해줘


ImportError: Could not import ddgs python package. Please install it with `pip install -U ddgs`.